In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import mlflow
import mlflow.xgboost
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns
import dtale
from default_risk.scripts.feature_cleaner import clean_importance_zero_and_negative_pfi
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

import gc



load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)
#the setup (we encapsulated that in a function for keep it constante during the joining to analize purely the gains of each table)
cv , hiperparams = get_baseline_setup() 



In [ ]:

#lets try with the main table without any treatment in the data.
application_train_df = pd.read_csv(cfg.RAW_DATA_DIR / "application_train.csv")

#minimun preparations necessary to be able to train the model with application_train
Y= application_train_df["TARGET"]
X= application_train_df.drop(columns=["TARGET"])
X.drop(columns=["SK_ID_CURR"],inplace=True)
X= cast_object_into_categoricals(X)



run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline")
#0.744 OOF auc is our baseline.

#cleaning memory
del application_train_df,X,Y  
gc.collect()

In [ ]:
#now let's repeat the set up with the version of the silver layer (Cleaned application_train)
#Therefore, this part need execute make_dataset first to generate the fold 01_cleaned

cleaned_application_train= pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
X,Y = prepare_columns(cleaned_application_train)
X= cast_object_into_categoricals(X)


run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline_cleaned_data")
#0.745 OOF auc. Just cleaning the data give us +0.1%, and with less risk of overfitting.


#cleaning memory
del cleaned_application_train,X,Y  
gc.collect()

In [ ]:
cleaned_application_train= pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
dtale.show(cleaned_application_train[:100])

In [ ]:
cleaned_application_train= pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
cleaned_application_train["ratio_debt_income"]= cleaned_application_train["amt_income_total"] / cleaned_application_train["amt_credit"]
cleaned_application_train["ratio_good_credit"]= cleaned_application_train["amt_goods_price"] / cleaned_application_train["amt_credit"]
cleaned_application_train["ratio_annuity_income"] = cleaned_application_train["amt_annuity"] / cleaned_application_train["amt_income_total"]
cleaned_application_train["ratio_days_employed_days_lived"]= cleaned_application_train["days_employed"] /(cleaned_application_train["days_birth"] * -1) 
cleaned_application_train["credit_duration"]= cleaned_application_train["amt_credit"] / cleaned_application_train["amt_annuity"]
cleaned_application_train["ext_1_x_2"] = cleaned_application_train["ext_source_1"] * cleaned_application_train["ext_source_2"]
cleaned_application_train["ext_2_x_3"] = cleaned_application_train["ext_source_2"] * cleaned_application_train["ext_source_3"]
cleaned_application_train["ext_1_x_3"] = cleaned_application_train["ext_source_1"] * cleaned_application_train["ext_source_3"]
cleaned_application_train["amount_of_scores_in_missing"] = cleaned_application_train["ext_source_1_is_missing"] + cleaned_application_train["ext_source_2_is_missing"] + cleaned_application_train["ext_source_3_is_missing"]
X,Y = prepare_columns(cleaned_application_train)
X= cast_object_into_categoricals(X)

cleaned_application_train.to_parquet(cfg.PROCESSED_DIR / "application_train_feature_engineering.parquet")

In [4]:
cleaned_application_train= pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")

cleaned_application_train["ratio_debt_income"] = (cleaned_application_train["amt_credit"] /  (cleaned_application_train["amt_income_total"]))
cleaned_application_train["ratio_debt_age"]= cleaned_application_train["amt_credit"] /(cleaned_application_train["days_birth"] * -1) 
cleaned_application_train["ratio_days_employed_days_lived"]= cleaned_application_train["days_employed"] /(cleaned_application_train["days_birth"] * -1) 
cleaned_application_train["kui_ratio"] =  np.where(cleaned_application_train["days_employed"] != 0, cleaned_application_train["amt_credit"] / ((cleaned_application_train["days_employed"] * -1) * cleaned_application_train["amt_income_total"]), 0)
cleaned_application_train["ratio_good_credit"]= cleaned_application_train["amt_goods_price"] / cleaned_application_train["amt_credit"]
cleaned_application_train["ratio_annuity_income"] = cleaned_application_train["amt_annuity"] / cleaned_application_train["amt_income_total"]
cleaned_application_train["credit_duration"]= cleaned_application_train["amt_credit"] / cleaned_application_train["amt_annuity"]
cleaned_application_train["ext_1_x_2"] = cleaned_application_train["ext_source_1"] * cleaned_application_train["ext_source_2"]
cleaned_application_train["ext_2_x_3"] = cleaned_application_train["ext_source_2"] * cleaned_application_train["ext_source_3"]
cleaned_application_train["ext_1_x_3"] = cleaned_application_train["ext_source_1"] * cleaned_application_train["ext_source_3"]




cleaned_application_train = pd.get_dummies(cleaned_application_train,columns=["organization_type"])
cleaned_application_train = pd.get_dummies(cleaned_application_train,columns=["education_type"])
#cleaned_application_train = pd.get_dummies(cleaned_application_train,columns=["occupation_type"])


#cleaned_application_train["amount_of_scores_in_missing"] = cleaned_application_train["ext_source_1_is_missing"] + cleaned_application_train["ext_source_2_is_missing"] + cleaned_application_train["ext_source_3_is_missing"]
ext_cols = ['ext_source_1', 'ext_source_2', 'ext_source_3']

building_features_names = [
    "APARTMENTS_AVG","BASEMENTAREA_AVG","YEARS_BEGINEXPLUATATION_AVG","YEARS_BUILD_AVG",
    "COMMONAREA_AVG","ELEVATORS_AVG","ENTRANCES_AVG","FLOORSMAX_AVG","FLOORSMIN_AVG",
    "LANDAREA_AVG","LIVINGAPARTMENTS_AVG","LIVINGAREA_AVG","NONLIVINGAPARTMENTS_AVG",
    "NONLIVINGAREA_AVG","APARTMENTS_MODE","BASEMENTAREA_MODE","YEARS_BEGINEXPLUATATION_MODE",
    "YEARS_BUILD_MODE","COMMONAREA_MODE","ELEVATORS_MODE","ENTRANCES_MODE","FLOORSMAX_MODE",
    "FLOORSMIN_MODE","LANDAREA_MODE","LIVINGAPARTMENTS_MODE","LIVINGAREA_MODE",
    "NONLIVINGAPARTMENTS_MODE","NONLIVINGAREA_MODE","APARTMENTS_MEDI","BASEMENTAREA_MEDI",
    "YEARS_BEGINEXPLUATATION_MEDI","YEARS_BUILD_MEDI","COMMONAREA_MEDI","ELEVATORS_MEDI",
    "ENTRANCES_MEDI","FLOORSMAX_MEDI","FLOORSMIN_MEDI","LANDAREA_MEDI","LIVINGAPARTMENTS_MEDI",
    "LIVINGAREA_MEDI","NONLIVINGAPARTMENTS_MEDI","NONLIVINGAREA_MEDI","FONDKAPREMONT_MODE",
    "HOUSETYPE_MODE","TOTALAREA_MODE","WALLSMATERIAL_MODE","EMERGENCYSTATE_MODE"
    ]

building_features_names = [col.lower() for col in building_features_names]

categorical_bldg = ['fondkapremont_mode', 'housetype_mode', 'wallsmaterial_mode', 'emergencystate_mode']

numeric_bldg = [col for col in building_features_names if col not in categorical_bldg]



# Agregaciones horizontales (axis=1)
cleaned_application_train["ext_source_mean"] = cleaned_application_train[ext_cols].mean(axis=1)
#cleaned_application_train["ext_source_max"] = cleaned_application_train[ext_cols].max(axis=1)
#cleaned_application_train["ext_source_min"] = cleaned_application_train[ext_cols].min(axis=1)
cleaned_application_train["ext_source_std"] = cleaned_application_train[ext_cols].std(axis=1)



#cleaned_application_train["debt_age_x_ext_source"] = np.where(  cleaned_application_train["ext_source_mean"] != 0,  cleaned_application_train["ratio_debt_age"] / cleaned_application_train["ext_source_mean"], 0)


# Agregaciones horizontales (axis=1)
cleaned_application_train["building_score_mean"] = cleaned_application_train[numeric_bldg].mean(axis=1)
cleaned_application_train["building_score_max"] = cleaned_application_train[numeric_bldg].max(axis=1)
cleaned_application_train["building_score_min"] = cleaned_application_train[numeric_bldg].min(axis=1)
cleaned_application_train["building_score_std"] = cleaned_application_train[numeric_bldg].std(axis=1)
cleaned_application_train["building_score_sum"] = cleaned_application_train[numeric_bldg].sum(axis=1)

cleaned_application_train["building_features_nan_count"] = cleaned_application_train[building_features_names].isnull().sum(axis=1)

#numeric_bldg.append("obs_30_cnt_social_circle")
cleaned_application_train= cleaned_application_train.drop(columns=numeric_bldg)
#cleaned_application_train= cleaned_application_train.drop(columns=["flag_document_5"] )


X,Y = prepare_columns(cleaned_application_train)
#X_CUTED=pd.DataFrame()
#X_CUTED[['own_car_age', 'ext_source_2', 'amt_goods_price', 'ext_source_3', 'amt_annuity', 'occupation_type', 'ext_source_1', 'days_id_publish', 'ext_2_x_3', 'days_birth', 'code_gender', 'ext_source_mean', 'kui_ratio', 'education_type_Higher education', 'credit_duration', 'ratio_good_credit']]= X[['own_car_age', 'ext_source_2', 'amt_goods_price', 'ext_source_3', 'amt_annuity', 'occupation_type', 'ext_source_1', 'days_id_publish', 'ext_2_x_3', 'days_birth', 'code_gender', 'ext_source_mean', 'kui_ratio', 'education_type_Higher education', 'credit_duration', 'ratio_good_credit']]
X= cast_object_into_categoricals(X)

cleaned_application_train.to_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline_cleaned_data")
#run_cv_tracked_mlflow_f_i(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"baseline_cleaned_data",run_pfi=True)
#0.753 OOF auc with basic feature engineering


#cleaning memory
#del cleaned_application_train,X,Y  
#gc.collect()

[0]	validation_0-auc:0.72640
[1]	validation_0-auc:0.73377
[2]	validation_0-auc:0.73697
[3]	validation_0-auc:0.73910
[4]	validation_0-auc:0.74176
[5]	validation_0-auc:0.74385
[6]	validation_0-auc:0.74511
[7]	validation_0-auc:0.74585
[8]	validation_0-auc:0.74792
[9]	validation_0-auc:0.74907
[10]	validation_0-auc:0.75019
[11]	validation_0-auc:0.75153
[12]	validation_0-auc:0.75233
[13]	validation_0-auc:0.75376
[14]	validation_0-auc:0.75413
[15]	validation_0-auc:0.75517
[16]	validation_0-auc:0.75532
[17]	validation_0-auc:0.75568
[18]	validation_0-auc:0.75624
[19]	validation_0-auc:0.75631
[20]	validation_0-auc:0.75673
[21]	validation_0-auc:0.75688
[22]	validation_0-auc:0.75669
[23]	validation_0-auc:0.75661
[24]	validation_0-auc:0.75694
[25]	validation_0-auc:0.75754
[26]	validation_0-auc:0.75746
[27]	validation_0-auc:0.75743
[28]	validation_0-auc:0.75741
[29]	validation_0-auc:0.75747
[30]	validation_0-auc:0.75814
[31]	validation_0-auc:0.75848
[32]	validation_0-auc:0.75870
[33]	validation_0-au

In [16]:
cleaned_application_train["diff"] =cleaned_application_train["obs_60_cnt_social_circle"] - cleaned_application_train[ "obs_30_cnt_social_circle"]
cleaned_application_train["diff"].describe()

count    306199.000000
mean         -0.016946
std           0.133033
min          -4.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           0.000000
Name: diff, dtype: float64

In [3]:
feature_permutation_df= pd.read_csv(cfg.ARTIFACTS_DIR / "pfi_cv_results_global.csv")
X_cleaned= clean_importance_zero_and_negative_pfi(feature_permutation_df,X)
run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X_cleaned,Y,experiment_name,"baseline_feature_permutation_cleaned")



eliminando ['amt_goods_price_is_missing', 'client_without_querys', 'flag_document_9', 'organization_type_Telecom', 'organization_type_XNA', 'organization_type_Other industry', 'own_car_age_is_missing', 'info_of_social_circule_is_missing', 'ext_source_1_is_missing', 'ext_source_2_is_missing', 'flag_emp_phone', 'ext_source_3_is_missing'] por importancia 0 o negativa en feature permutation
[0]	validation_0-auc:0.72640
[1]	validation_0-auc:0.73377
[2]	validation_0-auc:0.73697
[3]	validation_0-auc:0.73910
[4]	validation_0-auc:0.74176
[5]	validation_0-auc:0.74385
[6]	validation_0-auc:0.74511
[7]	validation_0-auc:0.74585
[8]	validation_0-auc:0.74792
[9]	validation_0-auc:0.74907
[10]	validation_0-auc:0.75019
[11]	validation_0-auc:0.75153
[12]	validation_0-auc:0.75233
[13]	validation_0-auc:0.75376
[14]	validation_0-auc:0.75413
[15]	validation_0-auc:0.75517
[16]	validation_0-auc:0.75532
[17]	validation_0-auc:0.75568
[18]	validation_0-auc:0.75624
[19]	validation_0-auc:0.75631
[20]	validation_0-au